# S4 · AndinaLog 03B · Notebook 1 · Diagnóstico de telemetría IoT

Este notebook trabaja **solo** con `andinalog_iot_telemetry.csv`. Lee la capa Bronze desde `datasets/AndinaLog_03B_Bronce/`, detecta problemas y conserva los diez campos originales. No convierte unidades, no imputa ni corrige datos. El tratamiento de los casos recuperables corresponde al notebook 2, después de aprobar sus reglas.

Cada ejecución reemplaza cuatro archivos en `S4/salidas/`:

1. `andinalog_iot_telemetry_diagnosticado.csv`: todas las filas, los campos originales y solo `fila_bronze`, `en_cuarentena` y `columnas_con_problemas`.
2. `andinalog_iot_telemetry_problemas.csv`: una fila por problema, con columna, código estable y evidencia.
3. `andinalog_iot_telemetry_cuarentena.csv`: extracto informativo de las filas marcadas.
4. `andinalog_iot_telemetry_reporte_calidad.csv`: conteos y huella SHA-256 del CSV de origen.


## 1 · Configuración y origen

En local, ejecuta el notebook desde cualquier carpeta dentro del proyecto. En Colab, monta Drive, selecciona `ENTORNO = "drive"` y ajusta `RUTA_PROYECTO_DRIVE` a la carpeta que contiene `datasets/` y `S4/`. Las salidas van a `S4/salidas/` en ese mismo entorno.


In [11]:
from pathlib import Path
import hashlib
import os
import re
import tempfile
import pandas as pd

ENTORNO = "local"  # "local" o "drive"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"  # ajustar si la carpeta real es otra
CARPETA_DATASETS = "AndinaLog_03B_Bronce"
NOMBRE_CSV = "andinalog_iot_telemetry.csv"
VERSION_DIAGNOSTICO = "GIAD-M3-S4-IOT-diagnostico-v2"

COLUMNAS_ORIGINALES = [
    "timestamp", "viaje_id", "order_id", "camion_id", "producto_id",
    "temperatura_cabina_c", "temp_unit", "humedad_cabina_pct",
    "desviacion_termica_flag", "desviacion_proximos_60min_flag",
]

def encontrar_raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets" / CARPETA_DATASETS).is_dir() and (carpeta / "S4").is_dir():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz del proyecto; ejecuta dentro de practicasNotebookColab.")

def configurar_rutas(entorno, ruta_drive):
    if entorno == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(ruta_drive)
    elif entorno == "local":
        raiz = encontrar_raiz_local()
    else:
        raise ValueError("ENTORNO debe ser 'local' o 'drive'")
    bronze = raiz / "datasets" / CARPETA_DATASETS / NOMBRE_CSV
    salidas = raiz / "S4" / "salidas"
    if not bronze.is_file():
        raise FileNotFoundError(f"No se encontró el CSV Bronze: {bronze}")
    return bronze, salidas

RUTA_BRONZE, DIRECTORIO_SALIDAS = configurar_rutas(ENTORNO, RUTA_PROYECTO_DRIVE)
print("Bronze:", RUTA_BRONZE)
print("Salidas:", DIRECTORIO_SALIDAS)


Bronze: c:\Users\remrodri\Github\practicasNotebookColab\datasets\AndinaLog_03B_Bronce\andinalog_iot_telemetry.csv
Salidas: c:\Users\remrodri\Github\practicasNotebookColab\S4\salidas


## 2 · Carga y contrato

La lectura mantiene todas las columnas como texto y los vacíos como cadenas vacías. Las conversiones numéricas y de fecha usadas para comprobar errores son temporales: no se exportan como valores transformados.


In [12]:
def cargar_bronze(ruta):
    huella = hashlib.sha256(ruta.read_bytes()).hexdigest()
    df = pd.read_csv(ruta, dtype="string", encoding="utf-8-sig", keep_default_na=False)
    return df, huella

def validar_esquema(df):
    if list(df.columns) != COLUMNAS_ORIGINALES:
        faltantes = sorted(set(COLUMNAS_ORIGINALES) - set(df.columns))
        extras = sorted(set(df.columns) - set(COLUMNAS_ORIGINALES))
        raise ValueError(f"Esquema inesperado. Faltantes: {faltantes}; extras: {extras}; orden: {list(df.columns)}")
    if not df.columns.is_unique:
        raise ValueError("Hay nombres de columnas duplicados")
    return df

df_bronze, HASH_BRONZE = cargar_bronze(RUTA_BRONZE)
validar_esquema(df_bronze)
print(f"Bronze: {len(df_bronze):,} filas × {len(df_bronze.columns)} columnas")
print("SHA-256:", HASH_BRONZE)
display(df_bronze.head())


Bronze: 28,920 filas × 10 columnas
SHA-256: c5f78ed802f8455dfc8eac91295954c38a159029e77b8ed748875fe2d8cf7467


,timestamp,viaje_id,order_id,camion_id,producto_id,temperatura_cabina_c,temp_unit,humedad_cabina_pct,desviacion_termica_flag,desviacion_proximos_60min_flag
0,2026-08-04 20:15:00,VIA-00001,ORD-2026-05710,CAM-12,PROD-052,-18.75,C,74.2,0,0
1,2026-08-04 20:45:00,VIA-00001,ORD-2026-05710,CAM-12,PROD-052,-18.33,C,65.0,0,0
2,2026-08-04 21:15:00,VIA-00001,ORD-2026-05710,CAM-12,PROD-052,-17.52,C,73.0,0,0
3,2026-08-04 21:45:00,VIA-00001,ORD-2026-05710,CAM-12,PROD-052,-17.96,C,85.4,0,0
4,2026-08-04 22:15:00,VIA-00001,ORD-2026-05710,CAM-12,PROD-052,-18.96,C,60.6,0,0


## 3 · Catálogo y reglas de diagnóstico

Los códigos son estables para que el notebook 2 pueda reconocer el problema exacto. `columna_afectada` puede nombrar una sola columna o una clave compuesta, como `viaje_id+timestamp`. Las reglas detectan; no deciden todavía cómo corregir.


In [13]:
CATALOGO_PROBLEMAS = pd.DataFrame([
    ("timestamp", "FECHA_INVALIDA", "No cumple el formato o no existe en el calendario"),
    ("viaje_id", "FALTANTE", "Identificador vacío"),
    ("order_id", "FALTANTE", "Identificador vacío"),
    ("camion_id", "FALTANTE", "Identificador vacío"),
    ("producto_id", "FALTANTE", "Identificador vacío"),
    ("viaje_id+timestamp", "DUPLICADO", "Clave de lectura repetida; se marca la aparición posterior"),
    ("temp_unit", "UNIDAD_NO_RECONOCIDA", "Unidad distinta de C o F"),
    ("temperatura_cabina_c", "FALTANTE", "Campo vacío"),
    ("temperatura_cabina_c", "NO_NUMERICA", "Valor no convertible a número"),
    ("humedad_cabina_pct", "FALTANTE", "Campo vacío"),
    ("humedad_cabina_pct", "NO_NUMERICA", "Valor no convertible a número"),
    ("humedad_cabina_pct", "FUERA_RANGO", "Humedad relativa fuera de 0 a 100%"),
    ("desviacion_termica_flag", "FLAG_INVALIDA", "Valor distinto de 0 o 1"),
    ("desviacion_proximos_60min_flag", "FLAG_INVALIDA", "Valor distinto de 0 o 1"),
], columns=["columna_afectada", "codigo_error", "criterio"])
display(CATALOGO_PROBLEMAS)

def registrar_problema(df, mascara, columna, codigo, evidencia=None):
    mascara = mascara.fillna(False).astype(bool)
    filas = df.loc[mascara, ["fila_bronze"]].copy()
    filas["columna_afectada"] = columna
    filas["codigo_error"] = codigo
    if evidencia is None:
        evidencia = df[columna] if columna in df.columns else pd.Series("", index=df.index, dtype="string")
    filas["valor_original"] = evidencia.loc[mascara].astype("string").to_numpy()
    return filas

def texto(df, columna):
    return df[columna].astype("string").str.strip()

def detectar_identificadores(df):
    return [registrar_problema(df, texto(df, col).eq(""), col, "FALTANTE")
            for col in ["viaje_id", "order_id", "camion_id", "producto_id"]]

def detectar_timestamps_y_duplicados(df):
    ts = texto(df, "timestamp")
    formato = ts.str.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}").fillna(False)
    fecha = pd.to_datetime(ts, format="%Y-%m-%d %H:%M:%S", errors="coerce")
    invalida = ~formato | fecha.isna()
    clave = pd.DataFrame({"viaje_id": texto(df, "viaje_id"), "timestamp": ts})
    duplicada = clave.duplicated(keep="first")
    evidencia_clave = texto(df, "viaje_id") + " + " + ts
    return [
        registrar_problema(df, invalida, "timestamp", "FECHA_INVALIDA"),
        registrar_problema(df, duplicada, "viaje_id+timestamp", "DUPLICADO", evidencia_clave),
    ]

def detectar_medicion(df, columna, rango=None):
    valor = texto(df, columna)
    numero = pd.to_numeric(valor, errors="coerce")
    hallazgos = [
        registrar_problema(df, valor.eq(""), columna, "FALTANTE"),
        registrar_problema(df, valor.ne("") & numero.isna(), columna, "NO_NUMERICA"),
    ]
    if rango is not None:
        minimo, maximo = rango
        hallazgos.append(registrar_problema(df, numero.notna() & ~numero.between(minimo, maximo), columna, "FUERA_RANGO"))
    return hallazgos

def detectar_unidad_y_flags(df):
    unidad = texto(df, "temp_unit").str.upper()
    hallazgos = [registrar_problema(df, ~unidad.isin(["C", "F"]), "temp_unit", "UNIDAD_NO_RECONOCIDA")]
    for col in ["desviacion_termica_flag", "desviacion_proximos_60min_flag"]:
        valor = pd.to_numeric(texto(df, col), errors="coerce")
        hallazgos.append(registrar_problema(df, ~valor.isin([0, 1]), col, "FLAG_INVALIDA"))
    return hallazgos

def diagnosticar(df_bronze):
    principal = df_bronze.copy(deep=True)
    principal.insert(0, "fila_bronze", range(1, len(principal) + 1))
    hallazgos = (
        detectar_identificadores(principal)
        + detectar_timestamps_y_duplicados(principal)
        + detectar_medicion(principal, "temperatura_cabina_c")
        + detectar_medicion(principal, "humedad_cabina_pct", rango=(0, 100))
        + detectar_unidad_y_flags(principal)
    )
    problemas = pd.concat(hallazgos, ignore_index=True)
    problemas = problemas.sort_values(["fila_bronze", "columna_afectada", "codigo_error"], kind="stable").reset_index(drop=True)
    problemas["version_diagnostico"] = VERSION_DIAGNOSTICO
    columnas_por_fila = problemas.groupby("fila_bronze")["columna_afectada"].agg(
        lambda valores: "|".join(dict.fromkeys(valores))
    )
    principal["columnas_con_problemas"] = principal["fila_bronze"].map(columnas_por_fila).fillna("")
    principal["en_cuarentena"] = principal["columnas_con_problemas"].ne("")
    return principal, problemas

df_diagnosticado, df_problemas = diagnosticar(df_bronze)
df_cuarentena = df_diagnosticado.loc[df_diagnosticado["en_cuarentena"]].copy()
print(f"Principal: {len(df_diagnosticado):,}; problemas: {len(df_problemas):,}; filas en cuarentena: {len(df_cuarentena):,}")
display(df_problemas.groupby(["columna_afectada", "codigo_error"]).size().rename("filas").reset_index())


,columna_afectada,codigo_error,criterio
0,timestamp,FECHA_INVALIDA,No cumple el formato o no existe en el calendario
1,viaje_id,FALTANTE,Identificador vacío
2,order_id,FALTANTE,Identificador vacío
3,camion_id,FALTANTE,Identificador vacío
4,producto_id,FALTANTE,Identificador vacío
5,viaje_id+timestamp,DUPLICADO,Clave de lectura repetida; se marca la aparici...
6,temp_unit,UNIDAD_NO_RECONOCIDA,Unidad distinta de C o F
7,temperatura_cabina_c,FALTANTE,Campo vacío
8,temperatura_cabina_c,NO_NUMERICA,Valor no convertible a número
9,humedad_cabina_pct,FALTANTE,Campo vacío


Principal: 28,920; problemas: 335; filas en cuarentena: 330


,columna_afectada,codigo_error,filas
0,humedad_cabina_pct,FALTANTE,100
1,humedad_cabina_pct,FUERA_RANGO,15
2,temp_unit,UNIDAD_NO_RECONOCIDA,5
3,temperatura_cabina_c,FALTANTE,80
4,timestamp,FECHA_INVALIDA,15
5,viaje_id+timestamp,DUPLICADO,120


## 4 · Reporte y comprobaciones antes de exportar

El reporte registra la huella SHA-256 para reconocer la versión exacta del CSV de origen. Los conteos de problemas pueden superar el número de filas en cuarentena porque una fila puede tener varios hallazgos.


In [14]:
def construir_reporte(df_bronze, principal, problemas, ruta, huella):
    conteos = problemas.groupby(["columna_afectada", "codigo_error"]).size()
    datos = [
        ("archivo_bronze", ruta.name),
        ("sha256_bronze", huella),
        ("version_diagnostico", VERSION_DIAGNOSTICO),
        ("filas_bronze", len(df_bronze)),
        ("filas_diagnosticadas", len(principal)),
        ("filas_en_cuarentena", int(principal["en_cuarentena"].sum())),
        ("filas_sin_cuarentena", int((~principal["en_cuarentena"]).sum())),
        ("problemas_detectados", len(problemas)),
    ]
    datos += [(f"{col}:{codigo}", int(total)) for (col, codigo), total in conteos.items()]
    return pd.DataFrame(datos, columns=["metrica", "valor"])

def validar_resultados(df_bronze, principal, problemas, cuarentena, reporte):
    assert list(principal.columns) == ["fila_bronze", *COLUMNAS_ORIGINALES, "columnas_con_problemas", "en_cuarentena"]
    pd.testing.assert_frame_equal(principal[COLUMNAS_ORIGINALES], df_bronze[COLUMNAS_ORIGINALES])
    assert len(principal) == len(df_bronze)
    assert principal["fila_bronze"].is_unique
    assert len(cuarentena) == int(principal["en_cuarentena"].sum())
    assert problemas["fila_bronze"].isin(principal["fila_bronze"]).all()
    assert problemas[["columna_afectada", "codigo_error"]].apply(tuple, axis=1).isin(
        CATALOGO_PROBLEMAS[["columna_afectada", "codigo_error"]].apply(tuple, axis=1)
    ).all()
    assert set(problemas["fila_bronze"]) == set(cuarentena["fila_bronze"])
    assert len(reporte) >= 8

reporte_calidad = construir_reporte(df_bronze, df_diagnosticado, df_problemas, RUTA_BRONZE, HASH_BRONZE)
validar_resultados(df_bronze, df_diagnosticado, df_problemas, df_cuarentena, reporte_calidad)
display(reporte_calidad)
print("Comprobaciones previas a la exportación: correctas")


,metrica,valor
0,archivo_bronze,andinalog_iot_telemetry.csv
1,sha256_bronze,c5f78ed802f8455dfc8eac91295954c38a159029e77b8e...
2,version_diagnostico,GIAD-M3-S4-IOT-diagnostico-v2
3,filas_bronze,28920
4,filas_diagnosticadas,28920
5,filas_en_cuarentena,330
6,filas_sin_cuarentena,28590
7,problemas_detectados,335
8,humedad_cabina_pct:FALTANTE,100
9,humedad_cabina_pct:FUERA_RANGO,15


Comprobaciones previas a la exportación: correctas


## 5 · Exportación reproducible

Los cuatro CSV se escriben primero como archivos temporales en `S4/salidas/` y se reemplazan con el mismo nombre al final. El CSV Bronze nunca se sobrescribe. Si se vuelve a ejecutar con la misma fuente y reglas, las salidas se actualizan en lugar de acumular versiones antiguas.


In [15]:
def exportar_salidas(directorio, tablas, ruta_bronze, huella_inicial):
    if hashlib.sha256(ruta_bronze.read_bytes()).hexdigest() != huella_inicial:
        raise RuntimeError("El CSV Bronze cambió durante la ejecución; no se exportarán resultados")
    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}
    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix=".tmp_iot_", dir=directorio,
                                             encoding="utf-8-sig", newline="", delete=False) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)
        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)
    return list(temporales)

tablas_salida = {
    "andinalog_iot_telemetry_diagnosticado.csv": df_diagnosticado,
    "andinalog_iot_telemetry_problemas.csv": df_problemas,
    "andinalog_iot_telemetry_cuarentena.csv": df_cuarentena,
    "andinalog_iot_telemetry_reporte_calidad.csv": reporte_calidad,
}
rutas_creadas = exportar_salidas(DIRECTORIO_SALIDAS, tablas_salida, RUTA_BRONZE, HASH_BRONZE)
for ruta in rutas_creadas:
    print(ruta)
print("Bronze intacta; salidas anteriores reemplazadas")


c:\Users\remrodri\Github\practicasNotebookColab\S4\salidas\andinalog_iot_telemetry_diagnosticado.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\salidas\andinalog_iot_telemetry_problemas.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\salidas\andinalog_iot_telemetry_cuarentena.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\salidas\andinalog_iot_telemetry_reporte_calidad.csv
Bronze intacta; salidas anteriores reemplazadas


## Siguiente etapa

El notebook 2 leerá el archivo diagnosticado y el detalle de problemas. El informe de S4 justifica tratamientos posibles, pero ninguna regla de curación está aprobada automáticamente por este diagnóstico. Una fila saldrá de cuarentena solo cuando todos sus problemas hayan sido resueltos y validados.
